# Experiment: Case-001 Doctor Handoff Sprint

Objective: prioritize parallel tasks after the doctor handoff so clinician review is not treated as a passive queue. This notebook uses public-safe labels only.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass


@dataclass(frozen=True)
class SprintTrack:
    """Public-safe representation of one two-week execution track."""

    name: str
    output: str
    unblock_score: int
    can_run_without_doctor: bool
    privacy_risk: int

    def priority_score(self) -> int:
        """Rank tracks by unblock value, independence, and privacy risk."""
        score = self.unblock_score

        if self.can_run_without_doctor:
            score += 2

        return score - self.privacy_risk

In [ ]:
tracks = [
    SprintTrack("clinician_validation", "corrected labels", 5, False, 1),
    SprintTrack("record_collection", "private source index", 5, True, 3),
    SprintTrack("transfusion_burden_table", "de-identified burden table", 5, True, 2),
    SprintTrack("blood_bank_packet", "immune and matching packet", 4, True, 2),
    SprintTrack("iron_organ_packet", "iron and organ-risk packet", 4, True, 2),
    SprintTrack("research_outreach", "qualified reviewer contacts", 3, True, 1),
    SprintTrack("candidate_rerank", "research-only rerank", 2, True, 1),
]

ranked_tracks = sorted(tracks, key=lambda track: (-track.priority_score(), track.name))

decision = {
    "sprint_label": "case001_doctor_handoff_sprint_active",
    "top_parallel_tracks": [track.name for track in ranked_tracks[:4]],
    "doctor_is_single_blocking_queue": False,
    "patient_relevance_status": "patient_relevance_blocked",
}

decision

## Result

The next work is clinician validation plus private record collection and de-identified burden calculation in parallel. Candidate reranking stays research-only until clinician review.
